In [ ]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "README.md").is_file() and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "README.md").is_file():
    raise FileNotFoundError("Run this notebook from the project directory or one of its subdirectories.")


# EXP18 — TimesNet (2D-Variation Temporal Inception) + V5 Physics + Exact 78h Evaluation

본 노트북은 완전 무결측 정제가 완료된 `train_final_physics_v5.csv`를 입력으로 사용하여 **TimesNet** 모델을 학습합니다.

### EXP18 핵심 설계
1. **TimesNet 아키텍처**: FFT 기반 2D 텐서 변환 및 2D Inception Conv를 통해 다중 주기 및 국소 피크 진폭을 손실 없이 포착
2. **V5 완전 시간 그리드 연동**: 236,304행 무결측 데이터셋으로 윈도우 유실 방지
3. **Exact 78h Peak Metric 일원화**: Feature Ablation, Early Stopping, Optuna HPO를 78시간 독립 피크 RMSE 지표로 최적화
4. **OOF 파일(`oof_exp18_timesnet.csv`) 및 `submission_exp18_timesnet.csv` 산출** (iTransformer와의 후속 앙상블 대비)

In [ ]:
# ============================================================
# 0. SETUP & PACKAGES
# ============================================================
!pip -q install optuna

from pathlib import Path
import gc
import json
import math
import random
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from sklearn.preprocessing import StandardScaler

import torch
import torch.nn as nn
import torch.fft
from torch.utils.data import Dataset, DataLoader

import optuna
from optuna.samplers import TPESampler

warnings.filterwarnings("ignore")

SEED = 42
def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything()

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("DEVICE:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
# ============================================================
# 1. CONFIG
# ============================================================
DATA_PATH = Path(PROJECT_ROOT / "data" / "processed" / "train_final_physics_v5.csv")
TEST_CONTEXT_PATH = Path(PROJECT_ROOT / "data" / "raw" / "test_context.parquet")
TEST_INDEX_PATH = Path(PROJECT_ROOT / "data" / "raw" / "test_index.csv")
SUBMISSION_PATH = Path(PROJECT_ROOT / "submission" / "submission_exp18_timesnet.csv")

EXP_DIR = PROJECT_ROOT / "artifacts" / "experiments" / "exp18_timesnet"
EXP_DIR.mkdir(parents=True, exist_ok=True)

TIME_COL = "time"
STATION_COL = "station"

STEP_MINUTES = 10
STEPS_PER_HOUR = 6

INPUT_LEN = 289
LEAD_HOURS = [3, 6, 9, 12, 18, 24]
LEAD_STEPS = [h * STEPS_PER_HOUR for h in LEAD_HOURS]
MAX_LEAD = max(LEAD_STEPS)
N_TARGETS = len(LEAD_STEPS)

TRAIN_RATIO = 0.80

ABLATION_MAX_SAMPLES = 25000
ABLATION_EPOCHS = 12
ABLATION_PATIENCE = 3

N_TRIALS = 30
OPTUNA_EPOCHS = 18
OPTUNA_PATIENCE = 4

FINAL_EPOCHS = 35
FINAL_PATIENCE = 6
NUM_WORKERS = 0

print("INPUT_LEN:", INPUT_LEN)
print("LEADS:", LEAD_HOURS)
print("TARGET DATA:", DATA_PATH)


In [ ]:
# ============================================================
# 2. LOAD V5 DATA
# ============================================================
df = pd.read_csv(DATA_PATH, parse_dates=[TIME_COL])
df = df.sort_values([STATION_COL, TIME_COL]).reset_index(drop=True)

if "hs_original_observed" not in df.columns:
    df["hs_original_observed"] = df["hs"].notna().astype(np.int8)

print("shape:", df.shape)
print("stations:", df[STATION_COL].unique())
print("period:", df[TIME_COL].min(), "~", df[TIME_COL].max())


In [ ]:
# ============================================================
# 3. FEATURE CANDIDATE SETS
# ============================================================
FEATURE_GROUPS = {
    "BASE_WAVE": ["hs", "tp", "hmax"],
    "WIND_BASIC": ["wspd", "gust", "u_wind", "v_wind"],
    "WAVE_MOMENTUM": ["hs_diff_1h", "hs_diff_3h", "hs_mean_6h", "hs_mean_12h", "hs_max_6h", "hs_max_12h"],
    "WAVE_DYNAMICS": ["wave_steepness", "wave_energy", "effective_wind_forcing", "u_wave", "v_wave"],
    "WIND_HISTORY": ["wspd_mean_6h", "wspd_mean_12h", "gust_max_6h", "gust_max_12h", "gust_minus_wspd"],
    "PRESSURE_TENDENCY": ["caph_change_3h", "caph_change_6h", "caph_change_12h"],
    "ALIGNMENT": ["wind_wave_alignment", "wind_wave_diff"],
    "ATMOS": ["airt", "relh", "caph"],
}

for k, cols in list(FEATURE_GROUPS.items()):
    FEATURE_GROUPS[k] = [c for c in cols if c in df.columns]

def uniq(seq):
    return list(dict.fromkeys(seq))

FEATURE_SETS = {}
FEATURE_SETS["BASE_WAVE"] = uniq(FEATURE_GROUPS["BASE_WAVE"])
FEATURE_SETS["BASE_WAVE_WIND"] = uniq(FEATURE_GROUPS["BASE_WAVE"] + FEATURE_GROUPS["WIND_BASIC"])
FEATURE_SETS["BASE_WAVE_WIND_HISTORY"] = uniq(FEATURE_GROUPS["BASE_WAVE"] + FEATURE_GROUPS["WIND_HISTORY"])
FEATURE_SETS["BASE_WAVE_MOMENTUM"] = uniq(FEATURE_GROUPS["BASE_WAVE"] + FEATURE_GROUPS["WAVE_MOMENTUM"])
FEATURE_SETS["BASE_WAVE_DYNAMICS"] = uniq(FEATURE_GROUPS["BASE_WAVE"] + FEATURE_GROUPS["WAVE_DYNAMICS"])
FEATURE_SETS["PHYSICS_OPTIMAL_11"] = uniq(["hs", "tp", "hmax", "wspd", "u_wind", "v_wind", "hs_diff_1h", "wave_energy", "effective_wind_forcing", "u_wave", "v_wave"])
FEATURE_SETS["WAVE_WIND"] = uniq(FEATURE_GROUPS["BASE_WAVE"] + FEATURE_GROUPS["WIND_BASIC"] + FEATURE_GROUPS["WAVE_DYNAMICS"])
FEATURE_SETS["STORM_PHYSICS"] = uniq(FEATURE_GROUPS["BASE_WAVE"] + FEATURE_GROUPS["WIND_BASIC"] + FEATURE_GROUPS["WIND_HISTORY"] + FEATURE_GROUPS["PRESSURE_TENDENCY"] + FEATURE_GROUPS["WAVE_DYNAMICS"])
FEATURE_SETS["ALL_FEATURES"] = uniq(sum(FEATURE_GROUPS.values(), []))

for name, cols in list(FEATURE_SETS.items()):
    FEATURE_SETS[name] = [c for c in cols if c in df.columns]

print("Feature Sets defined:", list(FEATURE_SETS.keys()))


In [ ]:
# ============================================================
# 4. SAMPLE BUILDING & DATASET DEFINITION
# ============================================================
STATION_TO_ID = {station: i for i, station in enumerate(sorted(df[STATION_COL].unique()))}
WINDOW_META_KEYS = ("station_id", "start_idx", "end_idx", "origin_hs", "origin_time")

def build_samples(frame, features, input_len=INPUT_LEN, lead_steps=LEAD_STEPS):
    station_id, start_idx, end_idx = [], [], []
    origin_hs, origin_time = [], []
    x_by_station, hs_by_station = {}, {}
    max_lead = max(lead_steps)

    for station, g in frame.groupby(STATION_COL, sort=False):
        g = g.sort_values(TIME_COL).reset_index(drop=True)
        sid = STATION_TO_ID[station]
        Xv = g.loc[:, features].to_numpy(dtype=np.float32, copy=True)
        hsv = g["hs"].to_numpy(dtype=np.float32, copy=True)
        obs = g["hs_original_observed"].to_numpy(dtype=np.int8, copy=True)
        times = g[TIME_COL].to_numpy(copy=True)
        x_by_station[sid] = Xv
        hs_by_station[sid] = hsv
        dt = pd.Series(g[TIME_COL]).diff().dt.total_seconds().div(60).to_numpy()

        for e in range(input_len - 1, len(g) - max_lead):
            s = e - input_len + 1
            if not np.all(dt[s + 1:e + 1] == STEP_MINUTES):
                continue
            if not np.isfinite(Xv[s:e + 1]).all():
                continue
            target_idx = np.asarray([e + step for step in lead_steps], dtype=np.int64)
            if not np.isfinite(hsv[target_idx]).all() or not np.all(obs[target_idx] == 1):
                continue
            if not np.isfinite(hsv[e]):
                continue

            station_id.append(sid)
            start_idx.append(s)
            end_idx.append(e)
            origin_hs.append(hsv[e])
            origin_time.append(times[e])

    return {
        "station_id": np.asarray(station_id, dtype=np.int64),
        "start_idx": np.asarray(start_idx, dtype=np.int64),
        "end_idx": np.asarray(end_idx, dtype=np.int64),
        "origin_hs": np.asarray(origin_hs, dtype=np.float32),
        "origin_time": np.asarray(origin_time, dtype="datetime64[ns]"),
        "X_by_station": x_by_station,
        "hs_by_station": hs_by_station,
        "source_frame": frame,
        "features": list(features),
        "n_features": len(features),
    }

def chronological_split(samples, train_ratio=TRAIN_RATIO):
    times = pd.Series(pd.to_datetime(samples["origin_time"]))
    source_tz = samples["source_frame"][TIME_COL].dt.tz
    current_tz = times.dt.tz

    if source_tz is not None:
        if current_tz is None:
            times = times.dt.tz_localize(source_tz)
        else:
            times = times.dt.tz_convert(source_tz)
    elif current_tz is not None:
        times = times.dt.tz_localize(None)

    cutoff = times.quantile(train_ratio)
    train_mask = (times <= cutoff).to_numpy()
    valid_mask = (times > cutoff).to_numpy()

    def subset(mask):
        part = {key: samples[key][mask] for key in WINDOW_META_KEYS}
        part.update({
            "X_by_station": samples["X_by_station"],
            "hs_by_station": samples["hs_by_station"],
            "source_frame": samples["source_frame"],
            "features": samples["features"],
            "n_features": samples["n_features"],
            "split_cutoff": cutoff,
        })
        return part

    return subset(train_mask), subset(valid_mask), cutoff

def limit_ablation_samples(samples, max_samples=ABLATION_MAX_SAMPLES):
    n = len(samples["end_idx"])
    if n <= max_samples:
        return samples
    keep_idx = np.linspace(0, n - 1, max_samples, dtype=np.int64)
    reduced = {key: samples[key][keep_idx] for key in WINDOW_META_KEYS}
    reduced.update({
        "X_by_station": samples["X_by_station"],
        "hs_by_station": samples["hs_by_station"],
        "source_frame": samples["source_frame"],
        "features": samples["features"],
        "n_features": samples["n_features"],
    })
    return reduced

def scale_samples(train, valid):
    scaler = StandardScaler()
    train_rows = train["source_frame"][TIME_COL] <= train["split_cutoff"]
    scaler.fit(train["source_frame"].loc[train_rows, train["features"]])
    for Xv in train["X_by_station"].values():
        scaler.transform(Xv, copy=False)
    return dict(train), dict(valid), scaler

class WaveDataset(Dataset):
    def __init__(self, samples):
        self.X_by_station = samples["X_by_station"]
        self.hs_by_station = samples["hs_by_station"]
        self.station_id = samples["station_id"]
        self.start_idx = samples["start_idx"]
        self.end_idx = samples["end_idx"]
        self.origin_hs = samples["origin_hs"]
        self.lead_steps = np.asarray(LEAD_STEPS, dtype=np.int64)

    def __len__(self):
        return len(self.end_idx)

    def __getitem__(self, idx):
        sid = int(self.station_id[idx])
        start, end = int(self.start_idx[idx]), int(self.end_idx[idx])
        X = torch.from_numpy(self.X_by_station[sid][start:end + 1])
        y = torch.from_numpy(self.hs_by_station[sid][end + self.lead_steps])
        return X, y, torch.tensor(self.origin_hs[idx], dtype=torch.float32), torch.tensor(sid)

def make_loader(samples, batch_size=128, shuffle=False):
    ds = WaveDataset(samples)
    return DataLoader(
        ds,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=NUM_WORKERS,
        pin_memory=False,
        drop_last=False,
    )


In [ ]:
# ============================================================
# 5. TimesNet MODEL DEFINITION & EXACT 78H METRIC
# ============================================================
class Inception_Block_2D(nn.Module):
    def __init__(self, in_channels, out_channels, num_kernels=3, init_weight=True):
        super().__init__()
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.num_kernels = num_kernels
        kernels = []
        for i in range(self.num_kernels):
            kernels.append(nn.Conv2d(in_channels, out_channels, kernel_size=2 * i + 1, padding=i))
        self.kernels = nn.ModuleList(kernels)
        if init_weight:
            self._initialize_weights()

    def _initialize_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)

    def forward(self, x):
        res_list = []
        for i in range(self.num_kernels):
            res_list.append(self.kernels[i](x))
        res = torch.stack(res_list, dim=-1).mean(-1)
        return res

class TimesBlock(nn.Module):
    def __init__(self, seq_len, pred_len, top_k, d_model, d_ff, num_kernels=3):
        super().__init__()
        self.seq_len = seq_len
        self.pred_len = pred_len
        self.k = top_k
        self.conv = nn.Sequential(
            Inception_Block_2D(d_model, d_ff, num_kernels=num_kernels),
            nn.GELU(),
            Inception_Block_2D(d_ff, d_model, num_kernels=num_kernels)
        )

    def forward(self, x):
        B, T, N = x.size()
        period_list, period_weight = self._get_period(x)
        res = []
        for i in range(self.k):
            period = period_list[i]
            if (self.seq_len) % period != 0:
                length = (((self.seq_len) // period) + 1) * period
                padding = torch.zeros([x.shape[0], (length - (self.seq_len)), x.shape[2]], device=x.device)
                out = torch.cat([x, padding], dim=1)
            else:
                length = self.seq_len
                out = x
            out = out.reshape(B, length // period, period, N).permute(0, 3, 1, 2).contiguous()
            out = self.conv(out)
            out = out.permute(0, 2, 3, 1).reshape(B, -1, N)
            res.append(out[:, :(self.seq_len), :])
        res = torch.stack(res, dim=-1)
        period_weight = nn.functional.softmax(period_weight, dim=1)
        period_weight = period_weight.unsqueeze(1).unsqueeze(1).repeat(1, T, N, 1)
        res = torch.sum(res * period_weight, -1)
        res = res + x
        return res

    def _get_period(self, x):
        xf = torch.fft.rfft(x, dim=1)
        frequency_list = abs(xf).mean(0).mean(-1)
        frequency_list[0] = 0
        _, top_list = torch.topk(frequency_list, self.k)
        top_list = top_list.detach().cpu().numpy()
        period = x.shape[1] // top_list
        return period, abs(xf).mean(-1)[:, top_list]

class TimesNet(nn.Module):
    def __init__(self, seq_len, n_features, pred_len, e_layers=2, d_model=64, d_ff=64, top_k=3, num_kernels=3, dropout=0.1):
        super().__init__()
        self.seq_len = seq_len
        self.pred_len = pred_len
        self.enc_embedding = nn.Linear(n_features, d_model)
        self.layer = e_layers
        self.model = nn.ModuleList([
            TimesBlock(seq_len, pred_len, top_k, d_model, d_ff, num_kernels)
            for _ in range(e_layers)
        ])
        self.layer_norm = nn.LayerNorm(d_model)
        self.predict_linear = nn.Linear(seq_len * d_model, pred_len)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        enc_out = self.enc_embedding(x)
        for i in range(self.layer):
            enc_out = self.layer_norm(self.model[i](enc_out))
        output = self.dropout(enc_out)
        output = output.reshape(output.shape[0], -1)
        output = self.predict_linear(output)
        return output

# Competition Aligned Weighted Loss (대회 공식 검증과 정합)
class CompetitionAlignedRMSELoss(nn.Module):
    def __init__(self, threshold=1.5, high_weight=1.0, low_weight=0.3):
        super().__init__()
        self.threshold = threshold
        self.high_weight = high_weight
        self.low_weight = low_weight

    def forward(self, pred, target, origin_hs):
        diff_sq = (pred - target) ** 2
        weights = torch.where(
            origin_hs >= self.threshold,
            torch.tensor(self.high_weight, device=pred.device),
            torch.tensor(self.low_weight, device=pred.device)
        ).unsqueeze(-1)
        weighted_mse = torch.sum(weights * diff_sq) / torch.sum(weights)
        return torch.sqrt(weighted_mse + 1e-6)

def rmse(y_true, y_pred):
    return float(np.sqrt(np.mean((y_true - y_pred) ** 2)))

def lead_rmse(y_true, y_pred):
    return {f"rmse_{h}h": rmse(y_true[:, i], y_pred[:, i]) for i, h in enumerate(LEAD_HOURS)}

def select_78h_separated_indices(times, station_ids, origin_hs, min_hours=78):
    selected = []
    times = pd.to_datetime(times)
    for sid in np.unique(station_ids):
        idx = np.where((station_ids == sid) & (origin_hs >= 1.5))[0]
        if len(idx) == 0:
            continue
        idx = idx[np.argsort(times[idx])]
        last_time = None
        for i in idx:
            t = times[i]
            if last_time is None or (t - last_time) >= pd.Timedelta(hours=min_hours):
                selected.append(i)
                last_time = t
    return np.asarray(selected, dtype=int)

def exact_competition_evaluation(y_true, y_pred, samples):
    idx = select_78h_separated_indices(samples["origin_time"], samples["station_id"], samples["origin_hs"], min_hours=78)
    if len(idx) == 0:
        return np.nan, 0
    return rmse(y_true[idx], y_pred[idx]), len(idx)

def evaluate_model(model, loader, samples_meta):
    model.eval()
    preds, ys, origins = [], [], []
    with torch.no_grad():
        for X, y, origin_hs, _ in loader:
            X, y = X.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
            pred = model(X)
            preds.append(pred.cpu().numpy())
            ys.append(y.cpu().numpy())
            origins.append(origin_hs.numpy())

    y_true = np.concatenate(ys)
    y_pred = np.concatenate(preds)
    origin_hs = np.concatenate(origins)

    overall = rmse(y_true, y_pred)
    exact_comp, exact_n = exact_competition_evaluation(y_true, y_pred, samples_meta)
    result = {
        "overall_rmse": overall,
        "exact_78h_comp_rmse": exact_comp,
        "exact_78h_n": exact_n,
        **lead_rmse(y_true, y_pred),
    }
    return result, y_true, y_pred

def train_one_model(train_samples, valid_samples, params, max_epochs, patience, verbose=True):
    seed_everything(SEED)
    batch_size = params.get("batch_size", 128)
    train_loader = make_loader(train_samples, batch_size=batch_size, shuffle=True)
    valid_loader = make_loader(valid_samples, batch_size=batch_size, shuffle=False)

    model = TimesNet(
        seq_len=INPUT_LEN,
        n_features=train_samples["n_features"],
        pred_len=N_TARGETS,
        e_layers=params["e_layers"],
        d_model=params["d_model"],
        d_ff=params.get("d_ff", params["d_model"]),
        top_k=params.get("top_k", 3),
        num_kernels=params.get("num_kernels", 3),
        dropout=params["dropout"],
    ).to(DEVICE)

    optimizer = torch.optim.AdamW(model.parameters(), lr=params["lr"], weight_decay=params["weight_decay"])
    criterion = CompetitionAlignedRMSELoss(threshold=1.5, high_weight=1.0, low_weight=0.3)

    best_state = None
    best_score = np.inf
    wait = 0
    history = []

    for epoch in range(1, max_epochs + 1):
        model.train()
        train_losses = []
        for X, y, origin_hs, _ in train_loader:
            X, y = X.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
            origin_hs = origin_hs.to(DEVICE, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            pred = model(X)
            loss = criterion(pred, y, origin_hs)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            train_losses.append(loss.item())

        metrics, _, _ = evaluate_model(model, valid_loader, valid_samples)
        score = metrics["exact_78h_comp_rmse"]
        if np.isnan(score):
            score = metrics["overall_rmse"]

        history.append({"epoch": epoch, "train_loss": float(np.mean(train_losses)), **metrics})
        if verbose:
            print(f"Epoch {epoch:02d} | train={np.mean(train_losses):.5f} | exact_78h={metrics['exact_78h_comp_rmse']:.5f} | overall={metrics['overall_rmse']:.5f}")

        if score < best_score:
            best_score = score
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            wait = 0
        else:
            wait += 1
        if wait >= patience:
            break

    if best_state is not None:
        model.load_state_dict(best_state)
    final_metrics, y_true, y_pred = evaluate_model(model, valid_loader, valid_samples)
    return model, final_metrics, pd.DataFrame(history), y_true, y_pred


In [ ]:
# ============================================================
# 6. FEATURE ABLATION (Exact 78h Ranking)
# ============================================================
FIXED_PARAMS = {
    "d_model": 64,
    "e_layers": 2,
    "top_k": 3,
    "dropout": 0.15,
    "lr": 3e-4,
    "weight_decay": 1e-5,
    "batch_size": 64,
}

ablation_records = []

for feature_name, features in FEATURE_SETS.items():
    print("\n" + "=" * 80)
    print(f"ABLATION: {feature_name} | {len(features)} features")

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    samples = build_samples(df, features)
    original_total = len(samples["end_idx"])
    if original_total < 1000:
        print("SKIP: too few samples")
        continue

    samples = limit_ablation_samples(samples, ABLATION_MAX_SAMPLES)
    train_s, valid_s, cutoff = chronological_split(samples, train_ratio=TRAIN_RATIO)
    train_s, valid_s, scaler = scale_samples(train_s, valid_s)
    del samples
    gc.collect()

    model, metrics, history, y_true, y_pred = train_one_model(
        train_s, valid_s,
        params=FIXED_PARAMS,
        max_epochs=ABLATION_EPOCHS,
        patience=ABLATION_PATIENCE,
        verbose=False,
    )

    record = {
        "feature_set": feature_name,
        "n_features": len(features),
        "original_total_n": original_total,
        **metrics,
    }
    ablation_records.append(record)

    del model, scaler, train_s, valid_s, history, y_true, y_pred
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

ablation_df = pd.DataFrame(ablation_records).sort_values(["exact_78h_comp_rmse", "overall_rmse"], ascending=[True, True]).reset_index(drop=True)
ablation_df.to_csv(EXP_DIR / "feature_ablation.csv", index=False)
display(ablation_df)


In [ ]:
# ============================================================
# 7. SELECT BEST FEATURE SET & PREPARE FULL DATA
# ============================================================
BEST_FEATURE_NAME = ablation_df.iloc[0]["feature_set"]
BEST_FEATURES = FEATURE_SETS[BEST_FEATURE_NAME]

print("BEST FEATURE SET (78h Optimal):", BEST_FEATURE_NAME)
print(f"SELECTED {len(BEST_FEATURES)} FEATURES:", BEST_FEATURES)

best_samples = build_samples(df, BEST_FEATURES)
train_samples, valid_samples, split_cutoff = chronological_split(best_samples)
train_samples, valid_samples, best_scaler = scale_samples(train_samples, valid_samples)

print(f"train samples: {len(train_samples['end_idx'])}")
print(f"valid samples: {len(valid_samples['end_idx'])}")


In [ ]:
# ============================================================
# 8. OPTUNA HPO (TimesNet)
# ============================================================
def objective(trial):
    params = {
        "d_model": trial.suggest_categorical("d_model", [32, 64, 128]),
        "e_layers": trial.suggest_int("e_layers", 2, 4),
        "top_k": trial.suggest_int("top_k", 2, 4),
        "dropout": trial.suggest_float("dropout", 0.05, 0.30),
        "lr": trial.suggest_float("lr", 5e-5, 8e-4, log=True),
        "weight_decay": trial.suggest_float("weight_decay", 1e-6, 1e-3, log=True),
        "batch_size": trial.suggest_categorical("batch_size", [64, 128]),
    }
    try:
        model, metrics, _, _, _ = train_one_model(
            train_samples, valid_samples, params=params, max_epochs=OPTUNA_EPOCHS, patience=OPTUNA_PATIENCE, verbose=False
        )
        score = metrics["exact_78h_comp_rmse"]
        if np.isnan(score):
            score = metrics["overall_rmse"]
        del model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        return score
    except RuntimeError as e:
        if "out of memory" in str(e).lower():
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            gc.collect()
            raise optuna.TrialPruned("CUDA OOM")
        raise

study = optuna.create_study(direction="minimize", sampler=TPESampler(seed=SEED), study_name="exp18_timesnet")
study.optimize(objective, n_trials=N_TRIALS, gc_after_trial=True)

trials_df = study.trials_dataframe()
trials_df.to_csv(EXP_DIR / "optuna_trials.csv", index=False)
with open(EXP_DIR / "best_params.json", "w", encoding="utf-8") as f:
    json.dump(study.best_params, f, indent=2, ensure_ascii=False)

print("BEST VALUE (Exact 78h Peak RMSE):", study.best_value)
print("BEST PARAMS:", study.best_params)


In [ ]:
# ============================================================
# 9. FINAL RETRAINING & OOF GENERATION
# ============================================================
BEST_PARAMS = dict(study.best_params)
final_model, final_metrics, final_history, y_true, y_pred = train_one_model(
    train_samples, valid_samples, params=BEST_PARAMS, max_epochs=FINAL_EPOCHS, patience=FINAL_PATIENCE, verbose=True
)

print("\n===== FINAL EXP18 (TimesNet) METRICS =====")
display(pd.Series(final_metrics).to_frame("value"))
final_history.to_csv(EXP_DIR / "final_history.csv", index=False)
pd.DataFrame([final_metrics]).to_csv(EXP_DIR / "final_metrics.csv", index=False)

# 앙상블용 OOF 예측값 저장
oof_df = pd.DataFrame(y_pred, columns=[f"pred_{h}h" for h in LEAD_HOURS])
oof_df["origin_hs"] = valid_samples["origin_hs"]
oof_df["origin_time"] = valid_samples["origin_time"]
oof_df["station_id"] = valid_samples["station_id"]
oof_df.to_csv(PROJECT_ROOT / "oof" / PROJECT_ROOT / "oof" / "oof_exp18_timesnet.csv", index=False)
print(f"OOF predictions saved: {PROJECT_ROOT / "oof" / PROJECT_ROOT / "oof" / "oof_exp18_timesnet.csv"}")

torch.save(
    {
        "model_state_dict": final_model.state_dict(),
        "features": BEST_FEATURES,
        "feature_set_name": BEST_FEATURE_NAME,
        "input_len": INPUT_LEN,
        "lead_hours": LEAD_HOURS,
        "params": BEST_PARAMS,
        "station_to_id": STATION_TO_ID,
        "metrics": final_metrics,
    },
    EXP_DIR / "best_model.pt",
)
joblib.dump(best_scaler, EXP_DIR / "scaler.pkl")
print("Saved EXP18 artifacts to:", EXP_DIR.resolve())


In [ ]:
# ============================================================
# 10. TEST INFERENCE & SUBMISSION CREATION (EXP18 TimesNet)
# ============================================================
BASE_FALLBACK_COLS = ["tp", "wspd", "gust", "wdir", "wvdir", "airt", "relh", "caph"]
station_medians = df.groupby("station")[BASE_FALLBACK_COLS].median()
global_medians = df[BASE_FALLBACK_COLS].median()

ratio_df = df[df["hs"].notna() & df["hmax"].notna() & (df["hs"] > 0)].copy()
ratio_df["hmax_hs_ratio"] = ratio_df["hmax"] / ratio_df["hs"]
ratio_df = ratio_df[ratio_df["hmax_hs_ratio"].between(1.0, 3.0)]
station_hmax_ratio = ratio_df.groupby("station")["hmax_hs_ratio"].median()
global_hmax_ratio = float(ratio_df["hmax_hs_ratio"].median())

def get_station_median(station, col):
    value = np.nan
    if station in station_medians.index and col in station_medians.columns:
        value = station_medians.loc[station, col]
    if not np.isfinite(value):
        value = global_medians[col]
    return float(value)

def get_hmax_ratio(station):
    if station in station_hmax_ratio.index:
        value = station_hmax_ratio.loc[station]
        if np.isfinite(value):
            return float(value)
    return global_hmax_ratio

def add_test_features_exp18(context):
    feature_frames = []
    for case_id, group in context.groupby("case_id", sort=False):
        g = group.sort_values("step_minute").copy()
        st = g["station"].iloc[0]

        for col in ["hs", "tp", "hmax"]:
            g.loc[g[col] <= 0, col] = np.nan
        g.loc[g["wspd"] < 0, "wspd"] = np.nan
        g.loc[g["gust"] < 0, "gust"] = np.nan
        g.loc[~g["relh"].between(0, 100), "relh"] = np.nan
        g.loc[~g["caph"].between(950, 1050), "caph"] = np.nan

        num_cols = ["hs", "tp", "hmax", "wspd", "gust", "airt", "relh", "caph"]
        g[num_cols] = g[num_cols].interpolate(method="linear", limit_direction="both").ffill().bfill()

        for col in ["wdir", "wvdir"]:
            g[col] = g[col].ffill().bfill()
            if g[col].isna().any():
                g[col] = g[col].fillna(get_station_median(st, col))

        g["wdir"] %= 360.0
        g["wvdir"] %= 360.0

        if g["hs"].isna().any():
            g["hs"] = g["hs"].ffill().bfill()

        ratio = get_hmax_ratio(st)
        hmax_missing = g["hmax"].isna() & g["hs"].notna()
        if hmax_missing.any():
            g.loc[hmax_missing, "hmax"] = g.loc[hmax_missing, "hs"] * ratio

        for col in ["tp", "wspd", "gust", "airt", "relh", "caph"]:
            if g[col].isna().any():
                g[col] = g[col].fillna(get_station_median(st, col))

        g["hs"] = g["hs"].clip(lower=0.01)
        g["tp"] = g["tp"].clip(lower=0.01)
        g["wspd"] = g["wspd"].clip(lower=0.0)
        g["gust"] = g["gust"].clip(lower=0.0)
        g["hmax"] = np.maximum(g["hmax"], g["hs"])
        g["gust"] = np.maximum(g["gust"], g["wspd"])

        wdir_rad = np.deg2rad(g["wdir"])
        wvdir_rad = np.deg2rad(g["wvdir"])
        g["u_wind"] = g["wspd"] * np.sin(wdir_rad)
        g["v_wind"] = g["wspd"] * np.cos(wdir_rad)
        g["u_wave"] = g["hs"] * np.sin(wvdir_rad)
        g["v_wave"] = g["hs"] * np.cos(wvdir_rad)

        diff = (g["wdir"] - g["wvdir"] + 180) % 360 - 180
        g["wind_wave_diff"] = np.abs(diff)
        g["wind_wave_alignment"] = np.cos(np.deg2rad(diff))

        g["hs_diff_1h"] = g["hs"] - g["hs"].shift(6)
        g["hs_diff_3h"] = g["hs"] - g["hs"].shift(18)
        g["hs_mean_6h"] = g["hs"].rolling(36, min_periods=1).mean()
        g["hs_mean_12h"] = g["hs"].rolling(72, min_periods=1).mean()
        g["hs_max_6h"] = g["hs"].rolling(36, min_periods=1).max()
        g["hs_max_12h"] = g["hs"].rolling(72, min_periods=1).max()

        g["wspd_mean_6h"] = g["wspd"].rolling(36, min_periods=1).mean()
        g["wspd_mean_12h"] = g["wspd"].rolling(72, min_periods=1).mean()
        g["gust_max_6h"] = g["gust"].rolling(36, min_periods=1).max()
        g["gust_max_12h"] = g["gust"].rolling(72, min_periods=1).max()
        g["gust_minus_wspd"] = g["gust"] - g["wspd"]

        g["caph_change_3h"] = g["caph"] - g["caph"].shift(18)
        g["caph_change_6h"] = g["caph"] - g["caph"].shift(36)
        g["caph_change_12h"] = g["caph"] - g["caph"].shift(72)

        wl = 1.56 * (g["tp"] ** 2)
        g["wave_steepness"] = g["hs"] / np.maximum(wl, 1.0)
        g["wave_energy"] = g["hs"] ** 2
        g["effective_wind_forcing"] = (g["wspd"] ** 2) * g["wind_wave_alignment"]

        g = g.replace([np.inf, -np.inf], np.nan)
        g[BEST_FEATURES] = g[BEST_FEATURES].bfill().ffill()
        feature_frames.append(g)

    return pd.concat(feature_frames, ignore_index=True)

test_context = pd.read_parquet(TEST_CONTEXT_PATH)
test_index = pd.read_csv(TEST_INDEX_PATH)
test_features = add_test_features_exp18(test_context)

case_order = test_index["case_id"].drop_duplicates().tolist()
windows = []
for case_id in case_order:
    grp = test_features[test_features["case_id"] == case_id].sort_values("step_minute")
    X = grp[BEST_FEATURES].to_numpy(dtype=np.float32)
    windows.append(X)

X_test_raw = np.stack(windows)
X_test = best_scaler.transform(X_test_raw.reshape(-1, len(BEST_FEATURES))).reshape(X_test_raw.shape).astype(np.float32)

final_model.eval()
pred_batches = []
INFER_BATCH_SIZE = 128
with torch.no_grad():
    for start in range(0, len(X_test), INFER_BATCH_SIZE):
        xb = torch.from_numpy(X_test[start:start + INFER_BATCH_SIZE]).to(DEVICE)
        pred = final_model(xb).detach().cpu().numpy()
        pred_batches.append(pred)

preds = np.concatenate(pred_batches, axis=0)

pred_rows = []
for case_id, row in zip(case_order, preds):
    for lead_h, val in zip(LEAD_HOURS, row):
        pred_rows.append({"case_id": case_id, "lead_h": lead_h, "hs_pred": float(val)})

submission = test_index.merge(pd.DataFrame(pred_rows), on=["case_id", "lead_h"], how="left", validate="one_to_one")
submission["hs_pred"] = submission["hs_pred"].clip(lower=0.0, upper=30.0)
submission.to_csv(SUBMISSION_PATH, index=False, encoding="utf-8")

print("\n" + "=" * 80)
print(f"EXP18 TIMESNET SUBMISSION SAVED: {SUBMISSION_PATH}")
print("=" * 80)
display(submission.head(12))
